# Compare SES-residualized and non-SES-residualized brain-behavior results

This notebook compares the brain-behavior results with and without SES/INR residualization for both analysis scales:

- **Full pMTG region analyses**: the non-vertexwise pMTG-region FC analyses used by the `brain_behavior_*` notebooks.
- **Vertexwise analyses**: the saved result tables from the `vertexwise_brain_behavior_*_ses_residualization.ipynb` notebooks.

Important conventions:

- Generated `*_full_*` FC columns are excluded from analysis dataframes before testing.
- Cognitive task analyses are limited to matched exploratory factor analysis (EFA) scores from `PCA_tasks.ipynb`; raw individual cognitive task scores are not tested here.
- Cognitive EFA tests are FDR-corrected within each analysis output. Paired SES-vs-no-SES comparisons use tests available in both models.
- Vertexwise FC residualization uses `STANDARD_FC_COVARIATES`, which include `mean_fd_0.20`; the SES-controlled vertexwise model appends INR to that covariate set. The full pMTG region section recomputes FC residuals from the region-level input tables used by the brain-behavior notebooks.


In [ ]:
from pathlib import Path
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats

from sklearn.linear_model import LinearRegression
from statsmodels.stats.multitest import multipletests

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='colorblind')
pd.set_option('display.max_columns', 120)
pd.set_option('display.width', 180)


In [ ]:
# Notebook-local analysis helpers.
STANDARD_FC_COVARIATES = [
    'demo_sex_v2',
    'interview_age',
    'site_id_l',
    'ehi1b',
    'mean_fd_0.20',
]

DEFAULT_CATEGORICAL_COVARIATES = {
    'demo_sex_v2',
    'site_id_l',
    'ehi1b',
}


def standardize_subject_id(subject_ids):
    return (
        subject_ids.astype(str)
        .str.replace('_', '', regex=False)
        .str.replace('^sub-', '', regex=True)
    )


def load_motion_qa(motion_qa_path, motion_column='mean_fd_0.20'):
    motion_qa = pd.read_csv(motion_qa_path)
    motion_qa = motion_qa[['src_subject_id', motion_column]].copy()
    motion_qa['src_subject_id'] = standardize_subject_id(motion_qa['src_subject_id'])
    motion_qa = motion_qa.drop_duplicates(subset=['src_subject_id'])
    motion_qa[motion_column] = pd.to_numeric(motion_qa[motion_column], errors='coerce')
    return motion_qa


def merge_motion_qa(df, motion_qa_path, motion_column='mean_fd_0.20', how='left'):
    motion_qa = load_motion_qa(motion_qa_path, motion_column=motion_column)

    merged = df.copy()
    merged['src_subject_id'] = standardize_subject_id(merged['src_subject_id'])
    if motion_column in merged.columns:
        existing_motion = merged.groupby('src_subject_id')[motion_column].first()
        merged = merged.drop(columns=[motion_column])
        merged = merged.merge(motion_qa, on='src_subject_id', how=how)
        merged[motion_column] = merged[motion_column].fillna(
            merged['src_subject_id'].map(existing_motion)
        )
    else:
        merged = merged.merge(motion_qa, on='src_subject_id', how=how)
    return merged


def complete_covariate_mask(df, covariates, categorical_covariates=DEFAULT_CATEGORICAL_COVARIATES):
    categorical_covariates = set(categorical_covariates)
    complete = pd.Series(True, index=df.index)

    for covariate in covariates:
        is_categorical = df[covariate].dtype == 'object' or covariate in categorical_covariates
        if is_categorical:
            complete &= df[covariate].notna()
        else:
            complete &= pd.to_numeric(df[covariate], errors='coerce').notna()

    return complete

def encode_regression_covariates(df, covariates, categorical_covariates=DEFAULT_CATEGORICAL_COVARIATES):
    encoded_parts = []
    categorical_covariates = set(categorical_covariates)

    for covariate in covariates:
        if df[covariate].dtype == 'object' or covariate in categorical_covariates:
            encoded_parts.append(
                pd.get_dummies(
                    df[covariate],
                    prefix=covariate,
                    drop_first=True,
                    dtype=float,
                )
            )
        else:
            encoded_parts.append(
                pd.to_numeric(df[covariate], errors='coerce').to_frame(covariate)
            )

    if not encoded_parts:
        return pd.DataFrame(index=df.index)
    return pd.concat(encoded_parts, axis=1)


def residualize_fc_profiles(df, fc_columns, covariates=STANDARD_FC_COVARIATES):
    df = df.copy()
    covariate_complete = complete_covariate_mask(df, covariates)

    validity_groups = {}
    for column in fc_columns:
        valid_idx = df[column].notnull() & covariate_complete
        key = valid_idx.to_numpy(dtype=np.bool_).tobytes()
        if key not in validity_groups:
            validity_groups[key] = (valid_idx, [])
        validity_groups[key][1].append(column)

    residual_frames = []
    for valid_idx, columns in validity_groups.values():
        output_columns = [column + '_resid' for column in columns]
        residuals = pd.DataFrame(np.nan, index=df.index, columns=output_columns)
        if valid_idx.sum() > 0:
            observed = df.loc[valid_idx, columns].to_numpy()
            covariate_matrix = encode_regression_covariates(
                df.loc[valid_idx],
                covariates,
            )
            if covariate_matrix.shape[1] == 0:
                predicted = np.tile(observed.mean(axis=0), (len(observed), 1))
            else:
                model = LinearRegression()
                model.fit(covariate_matrix, observed)
                predicted = model.predict(covariate_matrix)
            residuals.loc[valid_idx, output_columns] = observed - predicted
        residual_frames.append(residuals)

    if residual_frames:
        df = pd.concat([df, *residual_frames], axis=1)

    return df


def compute_correlations(df, measures, fc_columns, required_nonmissing=None):
    results = []
    required_nonmissing = list(required_nonmissing or [])

    for measure in measures:
        for column in fc_columns:
            analysis_columns = list(dict.fromkeys([column, measure, *required_nonmissing]))
            temp_df = df[analysis_columns].dropna()

            if (
                len(temp_df) < 2
                or temp_df[column].nunique(dropna=True) < 2
                or temp_df[measure].nunique(dropna=True) < 2
            ):
                r_value = np.nan
                p_value = np.nan
            else:
                r_value, p_value = stats.pearsonr(temp_df[column], temp_df[measure])

            results.append({
                'measure': measure,
                'col': column,
                'r': r_value,
                'p': p_value,
                'n': len(temp_df),
            })

    return pd.DataFrame(results)


def apply_multiple_comparison_corrections(results_df, alpha=0.05, fdr_group_col=None):
    if results_df.empty:
        return results_df.assign(
            p_bonf=pd.Series(dtype=float),
            sig_bonf=pd.Series(dtype=bool),
            p_fdr=pd.Series(dtype=float),
            sig_fdr=pd.Series(dtype=bool),
        )

    corrected = results_df.copy()
    n_tests = len(corrected)
    corrected['p_bonf'] = np.minimum(corrected['p'] * n_tests, 1.0)
    corrected['sig_bonf'] = corrected['p_bonf'] < alpha

    valid_p = corrected['p'].notna()
    corrected['p_fdr'] = np.nan
    corrected['sig_fdr'] = False

    if fdr_group_col is None:
        fdr_families = pd.Series('all', index=corrected.index)
    else:
        fdr_families = corrected[fdr_group_col].astype('string').fillna('<missing>')

    for family in fdr_families.unique():
        family_valid_p = valid_p & fdr_families.eq(family)
        if family_valid_p.any():
            reject, p_fdr, _, _ = multipletests(
                corrected.loc[family_valid_p, 'p'],
                alpha=alpha,
                method='fdr_bh',
            )
            corrected.loc[family_valid_p, 'p_fdr'] = p_fdr
            corrected.loc[family_valid_p, 'sig_fdr'] = reject

    return corrected


## Configuration

The region-level section mirrors the existing `brain_behavior_*` notebooks and excludes generated `*_full_*` network columns.

In [ ]:
SES_VARIABLE = 'inr'
SES_RESIDUALIZATION_COVARIATE = 'inr'
ALPHA = 0.05
TOP_N_VERTICES = 25

VERBAL_ABILITY_DIR = Path('/Users/emk/Documents/Documents - Ron Weasley V/Research/Verbal-Ability')
LOCAL_DATA_DIR = VERBAL_ABILITY_DIR / 'Final'
PCA_RESULTS_DIR = LOCAL_DATA_DIR / 'pca_results'
WRANGLED_DATA_PATH = LOCAL_DATA_DIR / 'wrangled_pMTG_FC_data_midb61_meanFC.csv'

REGION_WITHOUT_SES_PATH = LOCAL_DATA_DIR / 'midb61_meanFC_clusters_motion_resid_2026-07-08_11-19.csv'
REGION_WITH_SES_PATH = LOCAL_DATA_DIR / 'midb61_meanFC_clusters_motion_inr_resid_2026-07-07_22-48.csv'
MOTION_QA_PATH = LOCAL_DATA_DIR / 'motion_QA_results.csv'

VERTEX_RESULTS_DIR = LOCAL_DATA_DIR / 'vertexwise_brain_behavior_results'
OUTPUT_DIR = LOCAL_DATA_DIR / 'ses_residualization_comparison_results'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

NETWORK_RENAME = {
    'Aud': 'AUD',
    'CO': 'AMN',
    'Sal': 'SAL',
    'Tpole': 'TPOLE',
    'Vis': 'VIS',
}

print('Vertexwise FC covariates include mean FD:', 'mean_fd_0.20' in STANDARD_FC_COVARIATES)
print('STANDARD_FC_COVARIATES:', STANDARD_FC_COVARIATES)
print('Comparison outputs will be saved to:', OUTPUT_DIR)


## Helper functions

In [ ]:


def ses_model_for_residualization(residualize_for_ses):
    return 'with_ses_residualization' if residualize_for_ses else 'without_ses_residualization'


def efa_score_metadata(residualize_for_ses):
    ses_model = ses_model_for_residualization(residualize_for_ses)
    prefix = 'ses_cognitive' if residualize_for_ses else 'no_ses_cognitive'
    path = PCA_RESULTS_DIR / f'efa_cognitive_scores_{ses_model}.csv'
    return ses_model, prefix, path


def load_matched_efa_scores(residualize_for_ses):
    ses_model, prefix, path = efa_score_metadata(residualize_for_ses)

    efa_scores = pd.read_csv(path)

    efa_scores['src_subject_id'] = standardize_subject_id(efa_scores['src_subject_id'])
    efa_scores = efa_scores.drop_duplicates(subset='src_subject_id')
    cognitive_efa_measures = [
        col for col in efa_scores.columns if col.startswith(f'{prefix}_EF')
    ]

    return ses_model, cognitive_efa_measures, efa_scores[['src_subject_id', *cognitive_efa_measures]]


def split_network_side(network):
    label = str(network).removesuffix('_full')
    side = None
    for suffix in ('_left', '_right'):
        if label.endswith(suffix):
            side = suffix[1:]
            label = label[: -len(suffix)]
            break
    return label, side


def clean_network_label(network, hemisphere=None):
    base, side = split_network_side(network)
    base = NETWORK_RENAME.get(base, base)
    hemi = str(hemisphere).upper() if hemisphere is not None else None

    # Match the existing brain_behavior plotting convention: left pMTG VAN is labelled LANG.
    if base == 'VAN' and (hemi == 'L' or side == 'left'):
        return 'LANG'
    return base


def annotate_region_results(results, ses_model):
    annotated = results.copy()
    parsed = annotated['col'].map(parse_region_fc_column).apply(pd.Series)
    annotated = pd.concat([annotated, parsed], axis=1)
    annotated['analysis_scope'] = 'full_pmtg_region'
    annotated['ses_model'] = ses_model
    annotated['measure_family'] = 'cognitive_efa'
    annotated['abs_r'] = annotated['r'].abs()
    return annotated


def annotate_vertex_results(results, ses_model):
    annotated = results.copy()

    if 'source_fc_column' in annotated.columns:
        annotated['source_fc_column'] = annotated['source_fc_column'].map(strip_residual_suffixes)
    else:
        annotated['source_fc_column'] = annotated['col'].map(strip_residual_suffixes)

    parsed = annotated['source_fc_column'].map(parse_vertexwise_fc_column)
    parsed_df = pd.DataFrame(
        parsed.tolist(),
        columns=['network_raw_from_source', 'python_index_from_source', 'hemisphere_from_source'],
        index=annotated.index,
    )
    annotated = pd.concat([annotated, parsed_df], axis=1)

    if 'network' in annotated.columns:
        annotated['network_raw'] = annotated['network']
    else:
        annotated['network_raw'] = annotated['network_raw_from_source']

    if 'python_index' not in annotated.columns:
        annotated['python_index'] = annotated['python_index_from_source']
    if 'hemisphere' not in annotated.columns:
        annotated['hemisphere'] = annotated['hemisphere_from_source']

    annotated['network'] = [
        clean_network_label(network, hemi)
        for network, hemi in zip(annotated['network_raw'], annotated['hemisphere'])
    ]

    annotated['analysis_scope'] = 'vertexwise'
    annotated['ses_model'] = ses_model
    annotated['measure_family'] = 'cognitive_efa'
    annotated['abs_r'] = annotated['r'].abs()
    return annotated


def compare_model_summaries(summary, index_cols):
    wide = summary.pivot_table(
        index=index_cols,
        columns='ses_model',
        values=['n_tests', 'n_sig_fdr', 'prop_sig_fdr', 'mean_r', 'mean_abs_r'],
        aggfunc='first',
    )
    wide.columns = [f'{metric}_{model}' for metric, model in wide.columns]
    wide = wide.reset_index()

    for metric in ['n_sig_fdr', 'prop_sig_fdr', 'mean_r', 'mean_abs_r']:
        without_col = f'{metric}_without_ses_residualization'
        with_col = f'{metric}_with_ses_residualization'
        if without_col in wide.columns and with_col in wide.columns:
            wide[f'delta_{metric}_with_minus_without'] = wide[with_col] - wide[without_col]

    return wide


In [ ]:
def pair_ses_results(without_results, with_results, key_cols=None):
    if key_cols is None:
        key_cols = ['measure', 'source_fc_column']

    without_cog = without_results.loc[without_results['measure_family'].eq('cognitive_efa')].copy()
    with_cog = with_results.loc[with_results['measure_family'].eq('cognitive_efa')].copy()

    paired = without_cog.merge(
        with_cog,
        on=key_cols,
        suffixes=('_without_ses', '_with_ses'),
    )

    for col in ['analysis_scope', 'network', 'network_raw', 'pmtg_subregion', 'hemisphere', 'python_index']:
        without_col = f'{col}_without_ses'
        with_col = f'{col}_with_ses'
        if without_col in paired.columns:
            paired[col] = paired[without_col]
        elif with_col in paired.columns:
            paired[col] = paired[with_col]

    paired['r_without_ses'] = paired['r_without_ses']
    paired['r_with_ses'] = paired['r_with_ses']
    paired['abs_r_without_ses'] = paired['r_without_ses'].abs()
    paired['abs_r_with_ses'] = paired['r_with_ses'].abs()
    paired['delta_r'] = paired['r_with_ses'] - paired['r_without_ses']
    paired['delta_abs_r'] = paired['abs_r_with_ses'] - paired['abs_r_without_ses']
    paired['abs_delta_r'] = paired['delta_r'].abs()

    sig_without = paired['sig_fdr_without_ses'].fillna(False).astype(bool)
    sig_with = paired['sig_fdr_with_ses'].fillna(False).astype(bool)
    paired['fdr_gained_after_ses'] = (~sig_without) & sig_with
    paired['fdr_lost_after_ses'] = sig_without & (~sig_with)
    paired['fdr_retained_after_ses'] = sig_without & sig_with
    paired['fdr_non_significant_both'] = (~sig_without) & (~sig_with)

    return paired


def summarize_model_results(results, group_cols):
    summary = (
        results
        .groupby(group_cols + ['ses_model'], dropna=False)
        .agg(
            n_tests=('r', 'size'),
            n_sig_fdr=('sig_fdr', 'sum'),
            mean_r=('r', 'mean'),
            mean_abs_r=('abs_r', 'mean'),
            median_abs_r=('abs_r', 'median'),
            max_abs_r=('abs_r', 'max'),
        )
        .reset_index()
    )
    summary['prop_sig_fdr'] = summary['n_sig_fdr'] / summary['n_tests']
    return summary


def summarize_change(paired, group_cols):
    summary = (
        paired
        .groupby(group_cols, dropna=False)
        .agg(
            n_paired_tests=('delta_r', 'size'),
            n_sig_fdr_without_ses=('sig_fdr_without_ses', 'sum'),
            n_sig_fdr_with_ses=('sig_fdr_with_ses', 'sum'),
            n_fdr_gained_after_ses=('fdr_gained_after_ses', 'sum'),
            n_fdr_lost_after_ses=('fdr_lost_after_ses', 'sum'),
            n_fdr_retained_after_ses=('fdr_retained_after_ses', 'sum'),
            mean_r_without_ses=('r_without_ses', 'mean'),
            mean_r_with_ses=('r_with_ses', 'mean'),
            mean_abs_r_without_ses=('abs_r_without_ses', 'mean'),
            mean_abs_r_with_ses=('abs_r_with_ses', 'mean'),
            mean_delta_r=('delta_r', 'mean'),
            mean_delta_abs_r=('delta_abs_r', 'mean'),
            mean_abs_delta_r=('abs_delta_r', 'mean'),
            median_abs_delta_r=('abs_delta_r', 'median'),
            max_abs_delta_r=('abs_delta_r', 'max'),
        )
        .reset_index()
    )
    summary['delta_n_sig_fdr_with_minus_without'] = summary['n_sig_fdr_with_ses'] - summary['n_sig_fdr_without_ses']
    summary['delta_mean_abs_r_with_minus_without'] = summary['mean_abs_r_with_ses'] - summary['mean_abs_r_without_ses']
    summary['delta_mean_r_with_minus_without'] = summary['mean_r_with_ses'] - summary['mean_r_without_ses']
    return summary


def test_network_change_against_overall(paired):
    rows = []
    valid = paired.dropna(subset=['abs_delta_r', 'network']).copy()

    for analysis_scope, scope_df in valid.groupby('analysis_scope'):
        overall_mean = scope_df['abs_delta_r'].mean()
        for network, network_df in scope_df.groupby('network'):
            values = network_df['abs_delta_r'].dropna()
            other_values = scope_df.loc[~scope_df['network'].eq(network), 'abs_delta_r'].dropna()
            reference_mean = other_values.mean()

            if len(values) >= 2 and np.isfinite(reference_mean):
                t_stat, p_value = stats.ttest_1samp(values, popmean=reference_mean, nan_policy='omit')
            else:
                t_stat, p_value = np.nan, np.nan

            rows.append({
                'analysis_scope': analysis_scope,
                'network': network,
                'n_paired_tests': len(values),
                'network_mean_abs_delta_r': values.mean(),
                'overall_mean_abs_delta_r': overall_mean,
                'other_networks_mean_abs_delta_r': reference_mean,
                'mean_difference_vs_other_networks': values.mean() - reference_mean,
                't': t_stat,
                'p': p_value,
            })

    tests = pd.DataFrame(rows)
    tests['sig_nominal'] = tests['p'] < ALPHA
    return tests.sort_values(['analysis_scope', 'p', 'network'], na_position='last')




## Full pMTG region-level results

This recomputes the non-vertexwise pMTG-region correlations using the same region-level inputs and covariate logic as the brain–behavior notebooks. Generated `*_full_*` columns are excluded.

In [ ]:
def compute_region_brain_behavior_results(data_path, residualize_for_ses):
    ses_model, cognitive_efa_measures, efa_scores = load_matched_efa_scores(residualize_for_ses)
    df = pd.read_csv(data_path)
    df['src_subject_id'] = standardize_subject_id(df['src_subject_id'])
    df = merge_motion_qa(df, MOTION_QA_PATH, how='left')
    required_inr_cols = [SES_VARIABLE, 'inr_missing', 'poverty_line_2017']
    missing_inr_cols = [col for col in required_inr_cols if col not in df.columns]
    if missing_inr_cols:
        wrangled_inr_columns = [
            'src_subject_id',
            'demo_comb_income_v2',
            'demo_roster_v2',
            'income_median',
            *required_inr_cols,
        ]
        wrangled_inr = pd.read_csv(
            WRANGLED_DATA_PATH,
            usecols=lambda column: column in wrangled_inr_columns,
        )
        wrangled_inr['src_subject_id'] = standardize_subject_id(wrangled_inr['src_subject_id'])
        wrangled_inr = wrangled_inr.drop_duplicates(subset='src_subject_id')

        existing_inr_cols = [col for col in wrangled_inr.columns if col != 'src_subject_id' and col in df.columns]
        df = df.drop(columns=existing_inr_cols)
        df = df.merge(wrangled_inr, on='src_subject_id', how='left', validate='many_to_one')

    missing_inr_cols = [col for col in required_inr_cols if col not in df.columns]

    df = df.drop(columns=[col for col in cognitive_efa_measures if col in df.columns])
    pre_merge_rows = len(df)
    df = df.merge(
        efa_scores,
        on='src_subject_id',
        how='left',
    )

    # Only Fisher-z FC columns are tested; generated _full networks are excluded.
    full_fc_cols = [column for column in df.columns if '_fz' in str(column) and '_full' in str(column)]
    if full_fc_cols:
        df = df.drop(columns=full_fc_cols)

    raw_fc_cols = [
        col for col in df.columns
        if col.endswith('_fz') and '_full' not in col
    ]

    fc_covariates = list(STANDARD_FC_COVARIATES)
    if residualize_for_ses:
        fc_covariates.append(SES_RESIDUALIZATION_COVARIATE)

    missing_fc_covariates = [covariate for covariate in fc_covariates if covariate not in df.columns]

    existing_fc_resid_cols = [f'{col}_resid' for col in raw_fc_cols if f'{col}_resid' in df.columns]
    if existing_fc_resid_cols:
        df = df.drop(columns=existing_fc_resid_cols)

    df = residualize_fc_profiles(df, raw_fc_cols, covariates=fc_covariates)
    analysis_fc_cols = [f'{col}_resid' for col in raw_fc_cols]

    for measure in cognitive_efa_measures:
        df[measure] = pd.to_numeric(df[measure], errors='coerce')

    results = compute_correlations(
        df,
        cognitive_efa_measures,
        analysis_fc_cols,
        required_nonmissing=[SES_VARIABLE] if residualize_for_ses else None,
    )
    results = apply_multiple_comparison_corrections(results, alpha=ALPHA).drop(
        columns=['p_bonf', 'sig_bonf'],
        errors='ignore',
    )
    results = annotate_region_results(results, ses_model=ses_model)

    print(f'{ses_model}: {len(results):,} region-level EFA tests; {results["sig_fdr"].sum():,} FDR-significant tests')
    print(f'  Region FC columns tested: {len(analysis_fc_cols):,}')
    print(f'  Cognitive EFA factors tested: {cognitive_efa_measures}')
    print(f'  FC covariates: {fc_covariates}')
    if residualize_for_ses:
        print(f'  Missing raw INR values excluded from SES-residualized correlations: {df[SES_VARIABLE].isna().sum()}')
    else:
        print('  INR is not required for no-SES region-level correlations.')
    return results


region_without_ses = compute_region_brain_behavior_results(REGION_WITHOUT_SES_PATH, residualize_for_ses=False)
region_with_ses = compute_region_brain_behavior_results(REGION_WITH_SES_PATH, residualize_for_ses=True)

region_without_ses.to_csv(OUTPUT_DIR / 'full_pmtg_region_without_ses_residualization_results.csv', index=False)
region_with_ses.to_csv(OUTPUT_DIR / 'full_pmtg_region_with_ses_residualization_results.csv', index=False)
display(region_without_ses.head())
display(region_with_ses.head())


## Vertexwise results

This section reads the saved vertexwise result tables. Re-run both vertexwise brain-behavior notebooks first if you need the newest EFA-only FDR outputs.

In [ ]:
def load_vertexwise_results(slug):
    path = VERTEX_RESULTS_DIR / f'vertexwise_brain_behavior_{slug}_all.csv'

    residualize_for_ses = slug == 'with_ses_residualization'
    _, prefix, _ = efa_score_metadata(residualize_for_ses)
    ses_model = slug
    results = pd.read_csv(path)
    results = results.loc[results['measure'].astype(str).str.startswith(f'{prefix}_EF')].copy()

    if 'p_fdr' not in results.columns or 'sig_fdr' not in results.columns:
        results = apply_multiple_comparison_corrections(results, alpha=ALPHA).drop(
            columns=['p_bonf', 'sig_bonf'],
            errors='ignore',
        )

    results = results.drop(columns=['p_bonf', 'sig_bonf'], errors='ignore')
    results = annotate_vertex_results(results, ses_model=ses_model)

    print(f'{ses_model}: {len(results):,} vertexwise EFA tests; {results["sig_fdr"].sum():,} FDR-significant tests')
    return results


vertex_without_ses = load_vertexwise_results('without_ses_residualization')
vertex_with_ses = load_vertexwise_results('with_ses_residualization')

display(vertex_without_ses.head())
display(vertex_with_ses.head())


## FDR survival counts and mean relationship strength

`mean_abs_r` is used as the average relationship-strength metric; signed `mean_r` is also retained so directional shifts are visible.

In [ ]:
all_results = pd.concat(
    [region_without_ses, region_with_ses, vertex_without_ses, vertex_with_ses],
    ignore_index=True,
)

overall_model_summary = summarize_model_results(all_results, ['analysis_scope', 'measure_family'])
overall_model_comparison = compare_model_summaries(overall_model_summary, ['analysis_scope', 'measure_family'])
overall_model_comparison.to_csv(OUTPUT_DIR / 'ses_residualization_fdr_counts_and_mean_strength_overall.csv', index=False)
print('Overall FDR counts and mean relationship strength by analysis scope and measure family')
print('Saved:', OUTPUT_DIR / 'ses_residualization_fdr_counts_and_mean_strength_overall.csv')
display(overall_model_comparison.round(4))

network_model_summary = summarize_model_results(all_results, ['analysis_scope', 'measure_family', 'network'])
network_model_comparison = compare_model_summaries(network_model_summary, ['analysis_scope', 'measure_family', 'network'])
network_model_comparison.sort_values(['analysis_scope', 'measure_family', 'network']).to_csv(OUTPUT_DIR / 'ses_residualization_fdr_counts_and_mean_strength_by_network.csv', index=False)
print('Network-level FDR counts and mean relationship strength')
print('Saved:', OUTPUT_DIR / 'ses_residualization_fdr_counts_and_mean_strength_by_network.csv')
display(network_model_comparison.sort_values(['analysis_scope', 'measure_family', 'network']).round(4))


## Paired SES-vs-no-SES comparisons

The paired comparisons align the same cognitive measure and FC target across the two models. INR itself is not paired because it is not tested as an outcome in the SES-controlled model.

In [ ]:
paired_region = pair_ses_results(region_without_ses, region_with_ses)
paired_vertex = pair_ses_results(vertex_without_ses, vertex_with_ses)
paired_all = pd.concat([paired_region, paired_vertex], ignore_index=True)

paired_region.to_csv(OUTPUT_DIR / 'full_pmtg_region_paired_ses_comparison.csv', index=False)
paired_vertex.to_csv(OUTPUT_DIR / 'vertexwise_paired_ses_comparison.csv', index=False)

print(f'Paired full pMTG region cognitive EFA tests: {len(paired_region):,}')
print(f'Paired vertexwise cognitive EFA tests: {len(paired_vertex):,}')
display(paired_region.head())
display(paired_vertex.head())


In [ ]:
overall_change_summary = summarize_change(paired_all, ['analysis_scope'])
network_change_summary = summarize_change(paired_all, ['analysis_scope', 'network'])
measure_change_summary = summarize_change(paired_all, ['analysis_scope', 'measure'])

overall_change_summary.to_csv(OUTPUT_DIR / 'ses_residualization_change_summary_overall.csv', index=False)
print('Overall paired change summary')
print('Saved:', OUTPUT_DIR / 'ses_residualization_change_summary_overall.csv')
display(overall_change_summary.round(4))
network_change_summary.sort_values(['analysis_scope', 'mean_abs_delta_r'], ascending=[True, False]).to_csv(OUTPUT_DIR / 'ses_residualization_change_summary_by_network.csv', index=False)
print('Network-level paired change summary')
print('Saved:', OUTPUT_DIR / 'ses_residualization_change_summary_by_network.csv')
display(network_change_summary.sort_values(['analysis_scope', 'mean_abs_delta_r'], ascending=[True, False]).round(4))
measure_change_summary.sort_values(['analysis_scope', 'mean_abs_delta_r'], ascending=[True, False]).to_csv(OUTPUT_DIR / 'ses_residualization_change_summary_by_efa_factor.csv', index=False)
print('EFA factor-level paired change summary')
print('Saved:', OUTPUT_DIR / 'ses_residualization_change_summary_by_efa_factor.csv')
display(measure_change_summary.sort_values(['analysis_scope', 'mean_abs_delta_r'], ascending=[True, False]).round(4))


## No-SES FDR-significant results after SES control

This block focuses on paired cognitive EFA tests that were FDR-significant before SES residualization, then tracks whether each result remains FDR-significant and how its effect size changes after SES control.

In [ ]:
no_ses_fdr_sig = paired_all.loc[
    paired_all['sig_fdr_without_ses'].fillna(False).astype(bool)
].copy()

no_ses_fdr_sig['ses_fdr_status'] = np.where(
    no_ses_fdr_sig['sig_fdr_with_ses'].fillna(False).astype(bool),
    'retained_fdr_after_ses',
    'lost_fdr_after_ses',
)
no_ses_fdr_sig['delta_p_fdr'] = no_ses_fdr_sig['p_fdr_with_ses'] - no_ses_fdr_sig['p_fdr_without_ses']
no_ses_fdr_sig['percent_change_abs_r'] = np.where(
    no_ses_fdr_sig['abs_r_without_ses'].gt(0),
    100 * no_ses_fdr_sig['delta_abs_r'] / no_ses_fdr_sig['abs_r_without_ses'],
    np.nan,
)
no_ses_fdr_sig['weakened_after_ses'] = no_ses_fdr_sig['delta_abs_r'] < 0
no_ses_fdr_sig['direction_changed_after_ses'] = (
    np.sign(no_ses_fdr_sig['r_without_ses']) != np.sign(no_ses_fdr_sig['r_with_ses'])
)

no_ses_fdr_sig_summary = (
    no_ses_fdr_sig
    .groupby(['analysis_scope', 'ses_fdr_status'], dropna=False)
    .agg(
        n_tests=('delta_r', 'size'),
        mean_abs_r_without_ses=('abs_r_without_ses', 'mean'),
        mean_abs_r_with_ses=('abs_r_with_ses', 'mean'),
        mean_delta_abs_r=('delta_abs_r', 'mean'),
        mean_percent_change_abs_r=('percent_change_abs_r', 'mean'),
        mean_abs_delta_r=('abs_delta_r', 'mean'),
        median_abs_delta_r=('abs_delta_r', 'median'),
        max_abs_delta_r=('abs_delta_r', 'max'),
        mean_delta_p_fdr=('delta_p_fdr', 'mean'),
        n_weakened_after_ses=('weakened_after_ses', 'sum'),
        n_direction_changed_after_ses=('direction_changed_after_ses', 'sum'),
    )
    .reset_index()
)
no_ses_fdr_sig_summary['prop_of_no_ses_fdr_sig'] = np.nan
if not no_ses_fdr_sig_summary.empty:
    no_ses_fdr_sig_summary['prop_of_no_ses_fdr_sig'] = (
        no_ses_fdr_sig_summary['n_tests']
        / no_ses_fdr_sig_summary.groupby('analysis_scope')['n_tests'].transform('sum')
    )

no_ses_fdr_sig_network_summary = (
    no_ses_fdr_sig
    .groupby(['analysis_scope', 'network', 'ses_fdr_status'], dropna=False)
    .agg(
        n_tests=('delta_r', 'size'),
        mean_abs_r_without_ses=('abs_r_without_ses', 'mean'),
        mean_abs_r_with_ses=('abs_r_with_ses', 'mean'),
        mean_delta_abs_r=('delta_abs_r', 'mean'),
        mean_percent_change_abs_r=('percent_change_abs_r', 'mean'),
        mean_abs_delta_r=('abs_delta_r', 'mean'),
        median_abs_delta_r=('abs_delta_r', 'median'),
        max_abs_delta_r=('abs_delta_r', 'max'),
        n_weakened_after_ses=('weakened_after_ses', 'sum'),
        n_direction_changed_after_ses=('direction_changed_after_ses', 'sum'),
    )
    .reset_index()
)

detail_cols = [
    'analysis_scope',
    'ses_fdr_status',
    'measure',
    'source_fc_column',
    'network',
    'network_raw',
    'pmtg_subregion',
    'hemisphere',
    'python_index',
    'r_without_ses',
    'r_with_ses',
    'delta_r',
    'abs_r_without_ses',
    'abs_r_with_ses',
    'delta_abs_r',
    'abs_delta_r',
    'percent_change_abs_r',
    'p_without_ses',
    'p_with_ses',
    'p_fdr_without_ses',
    'p_fdr_with_ses',
    'delta_p_fdr',
    'n_without_ses',
    'n_with_ses',
    'weakened_after_ses',
    'direction_changed_after_ses',
]
detail_cols = [col for col in detail_cols if col in no_ses_fdr_sig.columns]
status_rank = {'lost_fdr_after_ses': 0, 'retained_fdr_after_ses': 1}
no_ses_fdr_sig_detail = (
    no_ses_fdr_sig
    .assign(ses_fdr_status_rank=no_ses_fdr_sig['ses_fdr_status'].map(status_rank))
    .sort_values(
        ['analysis_scope', 'ses_fdr_status_rank', 'abs_delta_r'],
        ascending=[True, True, False],
    )
    [detail_cols]
)

detail_path = OUTPUT_DIR / 'ses_residualization_no_ses_fdr_significant_detailed_changes.csv'
no_ses_fdr_sig_detail.to_csv(detail_path, index=False)
print(f'Paired cognitive EFA tests FDR-significant without SES residualization: {len(no_ses_fdr_sig_detail):,}')
print('Saved detailed no-SES FDR-significant changes:', detail_path)

no_ses_fdr_sig_summary.sort_values(['analysis_scope', 'ses_fdr_status']).to_csv(OUTPUT_DIR / 'ses_residualization_no_ses_fdr_significant_status_summary.csv', index=False)
print('No-SES FDR-significant paired tests: retained or lost after SES control')
print('Saved:', OUTPUT_DIR / 'ses_residualization_no_ses_fdr_significant_status_summary.csv')
display(no_ses_fdr_sig_summary.sort_values(['analysis_scope', 'ses_fdr_status']).round(4))
no_ses_fdr_sig_network_summary.sort_values(['analysis_scope', 'ses_fdr_status', 'n_tests', 'mean_abs_delta_r'], ascending=[True, True, False, False]).to_csv(OUTPUT_DIR / 'ses_residualization_no_ses_fdr_significant_by_network.csv', index=False)
print('No-SES FDR-significant paired tests by network')
print('Saved:', OUTPUT_DIR / 'ses_residualization_no_ses_fdr_significant_by_network.csv')
display(no_ses_fdr_sig_network_summary.sort_values(['analysis_scope', 'ses_fdr_status', 'n_tests', 'mean_abs_delta_r'], ascending=[True, True, False, False]).round(4))

display(no_ses_fdr_sig_detail.head(50).round(4))


## Which networks change more than the overall pattern?

For each analysis scope, this exploratory test compares each network's distribution of `|Δr|` against the mean `|Δr|` of all other networks using a one-sample t-test. These network-change p values are left nominal because the multiple-comparison correction in this workflow is reserved for cognitive EFA factor-FC tests.

In [ ]:
network_change_tests = test_network_change_against_overall(paired_all)
network_change_tests.to_csv(OUTPUT_DIR / 'ses_residualization_network_change_tests.csv', index=False)
print('Networks with SES-related |r| changes that differ from other networks')
print('Saved:', OUTPUT_DIR / 'ses_residualization_network_change_tests.csv')
display(network_change_tests.round(4))

significant_network_changes = network_change_tests.loc[network_change_tests['sig_nominal']].copy()
if significant_network_changes.empty:
    print(f'No networks showed nominal p < {ALPHA} |Δr| differences from the other-network reference.')
else:
    display(significant_network_changes.round(4))


In [ ]:
for analysis_scope, plot_df in network_change_summary.groupby('analysis_scope'):
    plot_df = plot_df.sort_values('mean_abs_delta_r', ascending=False)
    fig, ax = plt.subplots(figsize=(12, 5))
    sns.barplot(data=plot_df, x='network', y='mean_abs_delta_r', ax=ax)
    ax.set_title(f'Average SES-related magnitude of change by network: {analysis_scope}')
    ax.set_xlabel('Network')
    ax.set_ylabel('Mean |Δr| after controlling for SES')
    ax.tick_params(axis='x', rotation=45)
    plt.tight_layout()
    plt.show()


## Which vertices are most and least affected by SES control?

This section summarizes vertexwise SES sensitivity across all paired cognitive EFA factor tests. `mean_abs_delta_r` ranks vertices by their average change after SES control; `max_abs_delta_r` records the strongest single measure/network change observed at that vertex.

In [ ]:
vertex_pairs = paired_vertex.dropna(subset=['python_index', 'hemisphere', 'abs_delta_r']).copy()
vertex_pairs['python_index'] = vertex_pairs['python_index'].astype(int)

vertex_effect_summary = (
    vertex_pairs
    .groupby(['python_index', 'hemisphere'], dropna=False)
    .agg(
        n_paired_tests=('abs_delta_r', 'size'),
        n_networks=('network', 'nunique'),
        n_measures=('measure', 'nunique'),
        mean_abs_delta_r=('abs_delta_r', 'mean'),
        median_abs_delta_r=('abs_delta_r', 'median'),
        max_abs_delta_r=('abs_delta_r', 'max'),
        mean_delta_abs_r=('delta_abs_r', 'mean'),
        mean_abs_r_without_ses=('abs_r_without_ses', 'mean'),
        mean_abs_r_with_ses=('abs_r_with_ses', 'mean'),
    )
    .reset_index()
)

strongest_vertex_rows = vertex_pairs.loc[
    vertex_pairs.groupby(['python_index', 'hemisphere'])['abs_delta_r'].idxmax(),
    ['python_index', 'hemisphere', 'measure', 'network', 'source_fc_column', 'r_without_ses', 'r_with_ses', 'delta_r', 'abs_delta_r'],
].rename(columns={
    'measure': 'largest_change_measure',
    'network': 'largest_change_network',
    'source_fc_column': 'largest_change_source_fc_column',
    'r_without_ses': 'largest_change_r_without_ses',
    'r_with_ses': 'largest_change_r_with_ses',
    'delta_r': 'largest_change_delta_r',
    'abs_delta_r': 'largest_change_abs_delta_r',
})

vertex_effect_summary = vertex_effect_summary.merge(
    strongest_vertex_rows,
    on=['python_index', 'hemisphere'],
    how='left',
)

most_affected_vertices = vertex_effect_summary.nlargest(TOP_N_VERTICES, 'mean_abs_delta_r')
least_affected_vertices = vertex_effect_summary.nsmallest(TOP_N_VERTICES, 'mean_abs_delta_r')

vertex_effect_summary.to_csv(OUTPUT_DIR / 'ses_residualization_vertex_effect_summary.csv', index=False)
most_affected_vertices.to_csv(OUTPUT_DIR / 'ses_residualization_most_affected_vertices.csv', index=False)
least_affected_vertices.to_csv(OUTPUT_DIR / 'ses_residualization_least_affected_vertices.csv', index=False)

print('Most affected vertices')
display(most_affected_vertices.round(4))
print('Least affected vertices')
display(least_affected_vertices.round(4))


In [ ]:
vertex_network_effect_summary = (
    vertex_pairs
    .groupby(['network', 'python_index', 'hemisphere'], dropna=False)
    .agg(
        n_paired_tests=('abs_delta_r', 'size'),
        mean_abs_delta_r=('abs_delta_r', 'mean'),
        median_abs_delta_r=('abs_delta_r', 'median'),
        max_abs_delta_r=('abs_delta_r', 'max'),
        mean_delta_abs_r=('delta_abs_r', 'mean'),
    )
    .reset_index()
)

top_vertices_by_network = (
    vertex_network_effect_summary
    .sort_values(['network', 'mean_abs_delta_r'], ascending=[True, False])
    .groupby('network', group_keys=False)
    .head(10)
)
least_vertices_by_network = (
    vertex_network_effect_summary
    .sort_values(['network', 'mean_abs_delta_r'], ascending=[True, True])
    .groupby('network', group_keys=False)
    .head(10)
)

vertex_network_effect_summary.to_csv(OUTPUT_DIR / 'ses_residualization_vertex_network_effect_summary.csv', index=False)
top_vertices_by_network.to_csv(OUTPUT_DIR / 'ses_residualization_most_affected_vertices_by_network.csv', index=False)
least_vertices_by_network.to_csv(OUTPUT_DIR / 'ses_residualization_least_affected_vertices_by_network.csv', index=False)

print('Most affected vertices within each network')
display(top_vertices_by_network.round(4))
print('Least affected vertices within each network')
display(least_vertices_by_network.round(4))
